In [44]:
schema = [
    ('Age',            'numeric',     None),
    ('Income',         'numeric',     None),
    ('Experience',     'numeric',     None),
    ('Satisfaction',   'ordinal',     ['Low', 'Medium', 'High']),
    ('Performance',    'ordinal',     ['Low', 'Medium', 'High']),
    ('Job_Role',       'nominal',     None),
    ('Education',      'nominal',     None),
    ('City',           'nominal',     None),
    ('Marital_Status', 'nominal',     None),
    ('HasCar',         'sym_binary',  None),
    ('OwnsHouse',      'sym_binary',  None),
    ('DiseaseA',       'asym_binary', None),
    ('DiseaseB',       'asym_binary', None),
    ('DiseaseC',       'asym_binary', None),
]

dataFIle = 'data_matrix.csv'

print(f"Schema has {len(schema)} attributes:")
for name, kind, levels in schema:
    extra = f"  levels={levels}" if levels else ''
    print(f"  {name:<20} {kind}{extra}")

Schema has 14 attributes:
  Age                  numeric
  Income               numeric
  Experience           numeric
  Satisfaction         ordinal  levels=['Low', 'Medium', 'High']
  Performance          ordinal  levels=['Low', 'Medium', 'High']
  Job_Role             nominal
  Education            nominal
  City                 nominal
  Marital_Status       nominal
  HasCar               sym_binary
  OwnsHouse            sym_binary
  DiseaseA             asym_binary
  DiseaseB             asym_binary
  DiseaseC             asym_binary


In [45]:
import csv

def load_data(filepath, schema):
    obj_ids = []
    records = []

    with open(filepath, 'r') as f:
        reader = csv.reader(f)
        next(reader)

        for row in reader:
            obj_ids.append(row[0])
            parsed = []

            for col in range(1, len(row)):
                val  = row[col].strip()
                kind = schema[col - 1][1]

                if val == '':
                    parsed.append(None) 
                elif kind == 'numeric':
                    parsed.append(float(val))
                elif kind in ('sym_binary', 'asym_binary'):
                    parsed.append(int(val))
                else:
                    parsed.append(val)  
            records.append(parsed)

    return obj_ids, records


obj_ids, records = load_data(dataFIle, schema)

N = len(records)   
P = len(schema)       

print(f"Loaded {N} objects, {P} attributes each.")
print(f"IDs: {obj_ids}")

Loaded 12 objects, 14 attributes each.
IDs: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']


In [46]:
def preprocess(records, schema):
    N = len(records)
    P = len(schema)
    col_range = {}
    ordinal_map = {}

    for f in range(P):
        name, kind, levels = schema[f]

        if kind == "numeric":
            lo, hi = None, None
            for i in range(N):
                v = records[i][f]
                if v is not None:
                    if lo is None or v < lo:
                        lo = v
                    if hi is None or v > hi:
                        hi = v
            col_range[f] = (lo, hi)

        elif kind == "ordinal":
            M = len(levels)
            ordinal_map[f] = {levels[r]: r / (M - 1) for r in range(M)}

    return col_range, ordinal_map


col_range, ordinal_map = preprocess(records, schema)

print("Numeric ranges:")
for f, (lo, hi) in col_range.items():
    print(f"  {schema[f][0]:<15} min={lo}  max={hi}")

print("\nOrdinal z-scores:")
for f, mapping in ordinal_map.items():
    print(f"  {schema[f][0]:<15} {mapping}")

Numeric ranges:
  Age             min=22.0  max=55.0
  Income          min=35.0  max=100.0
  Experience      min=2.0  max=20.0

Ordinal z-scores:
  Satisfaction    {'Low': 0.0, 'Medium': 0.5, 'High': 1.0}
  Performance     {'Low': 0.0, 'Medium': 0.5, 'High': 1.0}


In [47]:
def attr_dissimilarity(va, vb, f, schema, col_range, ordinal_map):

    _, kind, _ = schema[f]

    if va is None or vb is None:
        return 0, 0.0

    if kind == "asym_binary" and va == 0 and vb == 0:
        return 0, 0.0

    if kind == "numeric":
        lo, hi = col_range[f]
        span = hi - lo
        diff = va - vb
        diff = diff if diff >= 0 else -diff
        d_f = 0.0 if span == 0 else diff / span

    elif kind == "ordinal":
        za = ordinal_map[f][va]
        zb = ordinal_map[f][vb]
        diff = za - zb
        d_f = diff if diff >= 0 else -diff

    else:
        d_f = 0.0 if va == vb else 1.0

    return 1, d_f


print("attr_dissimilarity() ready.")
print()

delta, df = attr_dissimilarity(
    records[0][0], records[1][0], 0, schema, col_range, ordinal_map
)
print(
    f"Age: Object1={records[0][0]}, Object2={records[1][0]}  →  delta={delta}, d_f={round(df,4)}"
)

attr_dissimilarity() ready.

Age: Object1=25.0, Object2=40.0  →  delta=1, d_f=0.4545


In [48]:
def dissimilarity(rec_a, rec_b, schema, col_range, ordinal_map):

    total_delta = 0.0
    total_d = 0.0

    for f in range(len(schema)):
        delta, d_f = attr_dissimilarity(
            rec_a[f], rec_b[f], f, schema, col_range, ordinal_map
        )
        total_delta = total_delta + delta
        total_d = total_d + (delta * d_f)

    if total_delta == 0:
        return None

    return total_d / total_delta


d = dissimilarity(records[0], records[1], schema, col_range, ordinal_map)
print(f"d(Object 1, Object 2) = {round(d, 4)}")

d(Object 1, Object 2) = 0.7384


In [49]:
def build_matrix(records, schema, col_range, ordinal_map):

    N = len(records)
    table = []
    for i in range(N):
        table.append([0.0] * N)

    for i in range(N):
        for j in range(i + 1, N):
            d = dissimilarity(records[i], records[j], schema, col_range, ordinal_map)
            val = round(d, 4) if d is not None else -1.0
            table[i][j] = val
            table[j][i] = val

    return table


matrix = build_matrix(records, schema, col_range, ordinal_map)
print("Matrix built successfully.")

Matrix built successfully.


In [50]:
def print_matrix(matrix, obj_ids):
    N = len(matrix)
    print(f"{'':>5}", end="")
    for oid in obj_ids:
        print(f"{oid:>8}", end="")
    print()
    print("-" * (5 + 8 * N))
    for i in range(N):
        print(f"{obj_ids[i]:>5}", end="")
        for j in range(N):
            print(f"{matrix[i][j]:>8.4f}", end="")
        print()


print_matrix(matrix, obj_ids)

            1       2       3       4       5       6       7       8       9      10      11      12
-----------------------------------------------------------------------------------------------------
    1  0.0000  0.7384  0.5391  0.4079  0.7381  0.4403  0.5968  0.3343  0.7649  0.3185  0.5839  0.6168
    2  0.7384  0.0000  0.8118  0.4472  0.3112  0.8576  0.4330  0.5708  0.3925  0.9365  0.2725  0.4618
    3  0.5391  0.8118  0.0000  0.7424  0.8758  0.2353  0.6993  0.3872  0.8127  0.2631  0.6511  0.7828
    4  0.4079  0.4472  0.7424  0.0000  0.6959  0.7507  0.4747  0.4262  0.5256  0.6259  0.3925  0.4744
    5  0.7381  0.3112  0.8758  0.6959  0.0000  0.8355  0.4848  0.7372  0.3819  0.9765  0.4171  0.3225
    6  0.4403  0.8576  0.2353  0.7507  0.8355  0.0000  0.8763  0.4186  0.8841  0.3335  0.5851  0.8333
    7  0.5968  0.4330  0.6993  0.4747  0.4848  0.8763  0.0000  0.7689  0.2952  0.7452  0.5732  0.3918
    8  0.3343  0.5708  0.3872  0.4262  0.7372  0.4186  0.7689  0.0000  0.8473  0.3

In [51]:
with open('dissimilarity_matrix.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Object'] + obj_ids)
    for i in range(N):
        writer.writerow([obj_ids[i]] + matrix[i])

print("Saved to dissimilarity_matrix.csv")

Saved to dissimilarity_matrix.csv
